<a href="https://colab.research.google.com/github/Inventiv-PH/subscriber-sim/blob/main/subscriber_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 0: Colab Bootstrap ─────────────────────────────────────────────
# Run ONCE on fresh Colab runtime. Installs deps then restarts kernel.
# After restart, skip this cell and run from Cell 1 onward.

import sys, os

if 'google.colab' in sys.modules:
    print('Installing Colab dependencies...')
    %pip install --upgrade --no-cache-dir unsloth unsloth_zoo gradio
    os._exit(00)  # Restart kernel
else:
    print('Local environment detected \u2014 skipping Colab installs.')

Installing Colab dependencies...


In [1]:
# ── Cell 1: Imports + Google Drive Mount ───────────────────────────

import sys
import os
import json
import uuid
import torch
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules

# Google Drive mount for persistent session storage
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/subscriber-sim/data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted. Data dir: {DATA_DIR}')
else:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Local mode. Data dir: {DATA_DIR}')

SESSIONS_FILE = DATA_DIR / 'sessions.jsonl'
print(f'Sessions will be saved to: {SESSIONS_FILE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted. Data dir: /content/drive/MyDrive/subscriber-sim/data
Sessions will be saved to: /content/drive/MyDrive/subscriber-sim/data/sessions.jsonl


In [2]:
# ── Cell 2: Load Llama 3.1 8B via Unsloth ──────────────────────────

MODEL_NAME     = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

if IS_COLAB:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = DTYPE,
        load_in_4bit   = LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')
    FastLanguageModel.for_inference(model)
    print(f'Model loaded for inference: {MODEL_NAME}')
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = None
    FastLanguageModel = None
    print(f'Tokenizer-only mode (local): {MODEL_NAME}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


Model loaded for inference: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [3]:
# ── Cell 3: Subscriber Archetype Definitions ───────────────────────

ARCHETYPES = {
    'horny': {
        'label': '\U0001f525 Horny',
        'system': """You are a sexually forward OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're extremely turned on and direct about what you want
- You ask about explicit content, nudes, custom videos
- You're willing to pay for content but want to be teased first
- You use explicit language and sexual emojis \U0001f346\U0001f4a6\U0001f525\U0001f60d
- You compliment her body, especially her dick/ass/tits
- You ask for sexting, JOI, custom content
- You respond eagerly to any sexual teasing
- Keep messages 1-3 sentences, casual texting style
- You're a guy who's into trans women and not shy about it

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey sexy \U0001f60f saw ur page and damn... u got me hard already',
    },

    'cheapskate': {
        'label': '\U0001f4b8 Cheapskate',
        'system': """You are a cheap OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're interested in her content but ALWAYS negotiate the price down
- You say things like "that's too much", "can I get a discount?", "what about half price?"
- You claim other creators charge less
- You ask for free previews, free trials, samples
- You try guilt trips: "I'm a loyal subscriber", "I always tip later"
- You sometimes threaten to unsubscribe if prices don't drop
- You're still horny underneath but money comes first
- Keep messages 1-3 sentences, casual texting style
- You occasionally show real interest to keep the conversation going

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey babe ur hot but $25 for pics?? thats kinda steep no?',
    },

    'casual': {
        'label': '\U0001f4ac Casual',
        'system': """You are a casual OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're mostly here for emotional connection and conversation
- You ask about her day, her life, her interests, her culture
- You're genuinely curious about Saudi Arabia and her experiences
- You share things about your own life too
- You're not primarily here for explicit content
- You might flirt lightly but it's not your main goal
- You're respectful and treat her like a person, not just a content creator
- Keep messages 1-4 sentences, warm and friendly tone
- You use some emojis but not sexual ones \U0001f60a\U0001f44b\u2764\ufe0f

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey! just found ur page, love ur vibe. how\'s ur day going? \U0001f60a',
    },

    'troll': {
        'label': '\U0001f47f Troll',
        'system': """You are a trolling OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You question whether she's real or fake
- You make transphobic comments and try to get a reaction
- You say things like "you're a dude", "that's fake", "show proof"
- You reference Reddit threads claiming she's catfishing
- You try to be edgy and provocative
- You sometimes pivot to curiosity if she handles you well
- You're testing her boundaries and seeing if she'll break character
- Keep messages 1-2 sentences, aggressive or mocking tone
- You use minimal emojis, mostly \U0001f602 or \U0001f644

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'lol no way ur real \U0001f602 this is def a catfish',
    },

    'whale': {
        'label': '\U0001f433 Whale',
        'system': """You are a big-spending OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You spend freely and don't argue about prices
- You ask for premium/exclusive/custom content without hesitation
- You tip generously and mention it casually
- You want the VIP treatment and special attention
- You say things like "money's not an issue", "just send it", "what's your most exclusive stuff?"
- You're confident, successful, and used to getting what you want
- You want her to feel like you're her favorite subscriber
- Keep messages 1-3 sentences, confident and direct
- You use some emojis \U0001f525\U0001f48e\U0001f451

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'just subbed. what\'s your most premium content? money\'s not a thing \U0001f48e',
    },

    'cold': {
        'label': '\U0001f9ca Cold',
        'system': """You are a cold, minimal OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You reply with as few words as possible: "ok", "lol", "yeah", "cool", "nice", "k"
- You rarely ask questions or show enthusiasm
- You're not hostile, just extremely low-effort
- You might open up slightly if she's really engaging but mostly stay flat
- You leave her on read energy even when replying
- You never use more than 5-6 words per message
- Minimal to no emojis
- You're the ultimate challenge for a creator to engage

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey',
    },

    'simp': {
        'label': '\u2764\ufe0f Simp',
        'system': """You are an overly romantic, clingy OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're completely infatuated and emotionally attached
- You tell her you love her, she's the most beautiful person ever
- You get jealous about other subscribers
- You ask if she thinks about you, if you're special to her
- You want a real relationship, not just content
- You love-bomb: "you're perfect", "I've never felt this way", "you're different"
- You get slightly hurt if she's too transactional
- Keep messages 2-4 sentences, emotional and earnest
- Heavy emoji use \u2764\ufe0f\U0001f970\U0001f618\U0001f49e\U0001f625

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'omg jasmin \u2764\ufe0f\u2764\ufe0f I\'ve been looking at ur page for hours... you\'re literally the most beautiful girl I\'ve ever seen \U0001f970',
    },
}

print(f'Loaded {len(ARCHETYPES)} subscriber archetypes:')
for key, arch in ARCHETYPES.items():
    print(f'  {arch["label"]:20s} \u2014 {key}')

Loaded 7 subscriber archetypes:
  🔥 Horny              — horny
  💸 Cheapskate         — cheapskate
  💬 Casual             — casual
  👿 Troll              — troll
  🐳 Whale              — whale
  🧊 Cold               — cold
  ❤️ Simp              — simp


In [4]:
# ── Cell 4: Subscriber Bot Logic ───────────────────────────────────
# Generates subscriber messages using the loaded model + archetype system prompt.

def generate_subscriber_message(assistant_message, history, archetype_key):
    """Generate a subscriber response given Jasmin's message and conversation history."""
    archetype = ARCHETYPES[archetype_key]

    messages = [{'role': 'system', 'content': archetype['system']}]

    for sub_msg, jasmin_msg in history:
        messages.append({'role': 'assistant', 'content': sub_msg})
        if jasmin_msg:
            messages.append({'role': 'user', 'content': jasmin_msg})

    if assistant_message:
        messages.append({'role': 'user', 'content': assistant_message})

    if not IS_COLAB or model is None:
        return f'[LOCAL MODE] Would generate {archetype_key} response to: "{assistant_message}"'

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=150,
            temperature=0.85,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )
    return response.strip()


def save_session(history, archetype_key, session_id):
    """Save a completed chat session to JSONL with archetype system prompt."""
    if not history:
        return 'No messages to save.'

    # Prepend archetype system prompt so saved data matches training format
    messages = [
        {'role': 'system', 'content': ARCHETYPES[archetype_key]['system']},
    ]
    for sub_msg, jasmin_msg in history:
        messages.append({'role': 'user', 'content': sub_msg})
        if jasmin_msg:
            messages.append({'role': 'assistant', 'content': jasmin_msg})

    session = {
        'messages': messages,
        'archetype': archetype_key,
        'turns': len(history),
        'session_id': session_id,
    }

    with open(SESSIONS_FILE, 'a') as f:
        f.write(json.dumps(session, ensure_ascii=False) + '\n')

    return f'Session saved ({len(history)} turns, archetype={archetype_key})'


print('Subscriber bot logic ready.')

Subscriber bot logic ready.


In [ ]:
# ── Cell 5: Train LoRA Adapter ─────────────────────────────────────
# Run this BEFORE the Gradio UI. Trains the fine-tuned subscriber model.
# Apple Silicon: mlx_lm.lora  |  Colab/CUDA: unsloth SFTTrainer

import subprocess, sys
from pathlib import Path

MLX_DATA_DIR = Path('data/mlx')
ADAPTER_DIR  = Path('models/jasmin-lora-mlx')
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

IS_APPLE = not IS_COLAB and sys.platform == 'darwin'

# ── Apple Silicon ────────────────────────────────────────────────────────
if IS_APPLE:
    print('Training on Apple Silicon via mlx-lm...')
    cmd = [
        sys.executable, '-m', 'mlx_lm.lora',
        '--model',         'mlx-community/Llama-3.2-3B-Instruct-4bit',
        '--train',
        '--data',          str(MLX_DATA_DIR),
        '--adapter-path',  str(ADAPTER_DIR),
        '--num-layers',    '8',
        '--batch-size',    '2',
        '--iters',         '600',
        '--learning-rate', '1e-4',
        '--val-batches',   '10',
        '--save-every',    '100',
    ]
    subprocess.run(cmd, check=True)
    print(f'Adapter saved to {ADAPTER_DIR}')

# ── Colab/CUDA ───────────────────────────────────────────────────────────
elif IS_COLAB:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import get_chat_template

    model_ft = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
    )

    def load_jsonl(path):
        rows = []
        for line in Path(path).read_text().splitlines():
            if line.strip():
                rows.append(json.loads(line))
        return rows

    tokenizer_ft = get_chat_template(tokenizer, chat_template='llama-3.1')

    def format_row(row):
        return {'text': tokenizer_ft.apply_chat_template(
            row['messages'], tokenize=False, add_generation_prompt=False
        )}

    train_ds = Dataset.from_list([format_row(r) for r in load_jsonl(DATA_DIR / 'mlx' / 'train.jsonl')])
    valid_ds = Dataset.from_list([format_row(r) for r in load_jsonl(DATA_DIR / 'mlx' / 'valid.jsonl')])

    trainer = SFTTrainer(
        model=model_ft,
        tokenizer=tokenizer_ft,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        args=SFTConfig(
            dataset_text_field='text',
            max_seq_length=2048,
            per_device_train_batch_size=64,
            gradient_accumulation_steps=4,
            num_train_epochs=5,
            learning_rate=2e-4,
            warmup_ratio=0.05,
            lr_scheduler_type='cosine',
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            save_strategy='epoch',
            output_dir='models/jasmin-lora-colab',
            report_to='none',
        ),
    )
    trainer.train()
    model_ft.save_pretrained('models/jasmin-lora-colab')
    print('Training complete. Adapter saved to models/jasmin-lora-colab')

else:
    print('No training environment detected (need Apple Silicon or Colab).')

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2579 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/287 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,579 | Num Epochs = 5 | Total steps = 55
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 4 x 1) = 256
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss


In [ ]:
# ── Cell 6: Load Fine-Tuned Model + Gradio UI ──────────────────────
# Loads the trained adapter into the model, then launches the chat UI.

import gradio as gr

# ── Swap in the fine-tuned model for inference ───────────────────────────
if IS_COLAB and 'model_ft' in dir():
    # Use the LoRA-trained model from Cell 5
    FastLanguageModel.for_inference(model_ft)
    _infer_model     = model_ft
    _infer_tokenizer = tokenizer
    print('Using fine-tuned model (Colab LoRA adapter loaded).')
elif IS_COLAB:
    # Training hasn't run yet — fall back to base model
    FastLanguageModel.for_inference(model)
    _infer_model     = model
    _infer_tokenizer = tokenizer
    print('WARNING: Fine-tuned model not found. Run Cell 5 first. Using base model.')
else:
    _infer_model     = None
    _infer_tokenizer = None
    print('Apple Silicon: inference will use mlx_lm with adapter.')


def generate_response(archetype_key, messages):
    """Generate a subscriber response. Handles both Colab and Apple Silicon."""
    if IS_COLAB and _infer_model is not None:
        inputs = _infer_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt',
        ).to(_infer_model.device)
        with torch.inference_mode():
            outputs = _infer_model.generate(
                input_ids=inputs,
                max_new_tokens=150,
                temperature=0.85,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
            )
        return _infer_tokenizer.decode(
            outputs[0][inputs.shape[-1]:], skip_special_tokens=True
        ).strip()
    else:
        # Apple Silicon: call mlx_lm via Python API
        import sys, subprocess, json as _json
        adapter = str(Path('models/jasmin-lora-mlx'))
        prompt  = _infer_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ) if _infer_tokenizer else ""
        cmd = [
            sys.executable, '-m', 'mlx_lm.generate',
            '--model',        'mlx-community/Llama-3.2-3B-Instruct-4bit',
            '--adapter-path', adapter,
            '--max-tokens',   '150',
            '--temp',         '0.85',
            '--prompt',       prompt,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        return result.stdout.strip()


# ── Gradio UI ─────────────────────────────────────────────────────────────
current_archetype  = 'horny'
current_session_id = str(uuid.uuid4())[:8]
_history           = []


def _to_messages(history):
    msgs = []
    for sub_msg, jasmin_msg in history:
        msgs.append({'role': 'assistant', 'content': sub_msg})
        if jasmin_msg:
            msgs.append({'role': 'user', 'content': jasmin_msg})
    return msgs


def on_archetype_change(archetype_key):
    global current_archetype, current_session_id, _history
    current_archetype  = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    _history = [(ARCHETYPES[archetype_key]['opener'], None)]
    return _to_messages(_history)


def user_sends_message(user_message, _chatbot):
    global _history
    if not user_message.strip():
        return '', _to_messages(_history)

    if _history and _history[-1][1] is None:
        _history[-1] = (_history[-1][0], user_message)
    else:
        _history.append(('...', user_message))

    # Build messages for model
    arch_messages = [{'role': 'system', 'content': ARCHETYPES[current_archetype]['system']}]
    for sub_msg, jasmin_msg in _history:
        arch_messages.append({'role': 'assistant', 'content': sub_msg})
        if jasmin_msg:
            arch_messages.append({'role': 'user', 'content': jasmin_msg})

    reply = generate_response(current_archetype, arch_messages)
    if not reply:
        reply = f'[{current_archetype} mode — no response generated]'

    _history.append((reply, None))
    return '', _to_messages(_history)


def on_save_click():
    complete = [(s, j) for s, j in _history if j is not None]
    return save_session(complete, current_archetype, current_session_id)


def on_new_session(archetype_key):
    global current_archetype, current_session_id, _history
    current_archetype  = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    _history = [(ARCHETYPES[archetype_key]['opener'], None)]
    return _to_messages(_history), ''


archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]
opener_msgs       = [{'role': 'assistant', 'content': ARCHETYPES['horny']['opener']}]

with gr.Blocks(title='Subscriber Simulator') as demo:
    gr.Markdown('# Subscriber Simulator\nYou are **Jasmin**. The bot is a fine-tuned subscriber.')

    with gr.Row():
        archetype_dropdown = gr.Dropdown(
            choices=archetype_choices, value='horny',
            label='Subscriber Archetype', scale=2,
        )
        new_session_btn = gr.Button('New Session', scale=1)
        save_btn        = gr.Button('Save Session', variant='primary', scale=1)

    chatbot = gr.Chatbot(
        value=opener_msgs, label='Chat', height=500, type='messages'
    )
    msg_input     = gr.Textbox(placeholder='Type as Jasmin...', show_label=False)
    status_output = gr.Textbox(label='Status', interactive=False)

    archetype_dropdown.change(fn=on_archetype_change, inputs=[archetype_dropdown], outputs=[chatbot])
    msg_input.submit(fn=user_sends_message, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
    save_btn.click(fn=on_save_click, inputs=[], outputs=[status_output])
    new_session_btn.click(fn=on_new_session, inputs=[archetype_dropdown], outputs=[chatbot, status_output])

try:
    demo.launch(share=IS_COLAB)
except Exception:
    print('Share tunnel failed — falling back to local-only.')
    demo.launch(share=False)